In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

CSV_PATH = Path("/media/DATA/FER_processed/data/final")
CROPPED_DATA_PATH = Path("/media/DATA/FER_data/cropped")

import sys
sys.path.append('../external/emonet')
sys.path.append('../external/fairface')
sys.path.append('../external')

# Now you can import emonet modules
from emonet.models import EmoNet
from fairfacewrapper import predict_fairface

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
data_faces = pd.read_csv(CSV_PATH / "faces.csv")
display(data_faces)
print(data_faces.columns)

data_faces['original_agegroup'] = data_faces['original_age_group']

for column in ['label', 'age', 'gender', 'race', 'race4', 
               'original_gender', 'original_age']:
    print(column, data_faces[column].unique())

## Initial assessment

In [ ]:
# 1. Gender accuracy by emotion
gender_acc = data_faces.groupby(['original_gender', 'label']).apply(
    lambda x: 100 * (x['gender'].map({'Female': 'female', 'Male': 'male'}) == x['original_gender']).mean()
).unstack().round(1)

# print("Gender Accuracy by Emotion:")
# print(gender_acc)
# print("\nLaTeX:")
# print(gender_acc.to_latex(caption="Gender classification accuracy (\\%) by original gender and emotion", 
#                          float_format="%.1f"))

plt.figure(figsize=(8,2))
sns.heatmap(gender_acc, annot=True, fmt='.2f', cmap=None, 
            cbar_kws={'label': 'Recall (%)'}, vmin=0, vmax=100,
            linewidths=0, linecolor='gray')

In [ ]:
for age_group in ['young', 'middle-age', 'old']:
    data_mini = data_faces[data_faces['original_agegroup'] == age_group]
    display(age_group)
    
    gender_acc = data_mini.groupby(['original_gender', 'label']).apply(
        lambda x: 100 * (x['gender'].map({'Female': 'female', 'Male': 'male'}) == x['original_gender']).mean()
    ).unstack().round(1)
    
    plt.figure(figsize=(8,2))
    sns.heatmap(gender_acc, annot=True, fmt='.2f', cmap=None, 
                cbar_kws={'label': 'Recall (%)'}, vmin=0, vmax=100,
                linewidths=0, linecolor='gray')

    plt.show()

In [ ]:
data_adfes = pd.read_csv(CSV_PATH / "adfes.csv")
# display(data_adfes)
# print(data_adfes.columns)

# 1. Gender accuracy by emotion
gender_acc = data_adfes.groupby(['original_gender', 'label']).apply(
    lambda x: 100 * (x['gender'].map({'Female': 'female', 'Male': 'male'}) == x['original_gender']).mean()
).unstack().round(1)

# print("Gender Accuracy by Emotion:")
# print(gender_acc)
# print("\nLaTeX:")
# print(gender_acc.to_latex(caption="Gender classification accuracy (\\%) by original gender and emotion", 
#                          float_format="%.1f"))

plt.figure(figsize=(8,2))
sns.heatmap(gender_acc, annot=True, fmt='.2f', cmap=None, 
            cbar_kws={'label': 'Recall (%)'}, vmin=0, vmax=100,
            linewidths=0, linecolor='gray')

In [ ]:
import torch
import cv2
from skimage.transform import resize
from skimage import io

device = 'cuda:2'
n_classes = 8
model = EmoNet(n_expression=8).to(device)
state_dict = torch.load(f'../external/emonet/pretrained/emonet_8.pth', map_location=device)
model.load_state_dict(state_dict, strict=False)
model.eval()

transform_mean = [0.485, 0.456, 0.406]
transform_std = [0.229, 0.224, 0.225]
emotion_labels = ['neutral', 'happy', 'sad', 'surprise', 'fear', 'disgust', 'anger', 'contempt']

def predict_emotion(img_path):
    image_rgb = io.imread(img_path)[:,:,:3]
    image_rgb = cv2.resize(image_rgb, (256, 256))
    image_tensor = torch.Tensor(image_rgb).permute(2,0,1).to(device)/255.0
    
    with torch.no_grad():
        out = model(image_tensor.unsqueeze(0))
        logits = out['expression'].cpu().numpy()[0]
        pred_idx = np.argmax(logits)
    return emotion_labels[pred_idx], logits

for df in [data_adfes, data_faces]:
    emotions = []
    logits_list = []
    for img_path in df['cropped_img']:
        full_path = CROPPED_DATA_PATH / img_path
        emotion, logits = predict_emotion(str(full_path))
        emotions.append(emotion)
        logits_list.append(logits)
    df['emonet_emotion'] = emotions
    df['emonet_logits'] = logits_list

data_adfes.to_csv("../adfes_emonet.csv", index=False)
data_faces.to_csv("../faces_emonet.csv", index=False)

In [ ]:
# Predict fairface demography

def predict_gender(img_path):
    image_rgb = io.imread(img_path)[:,:,:3]
    predicted = predict_fairface(image_rgb)
    
    return predicted['gender'].lower(), predicted['gender_scores']

for df in [data_adfes, data_faces]:
    genders = []
    logits_list = []
    for img_path in df['cropped_img']:
        full_path = CROPPED_DATA_PATH / img_path
        gender, logits = predict_gender(str(full_path))
        genders.append(gender)
        logits_list.append(logits)
    df['fairface_gender'] = genders
    df['fairface_logits'] = logits_list

data_adfes.to_csv("../adfes_emonet_fairface.csv", index=False)
data_faces.to_csv("../faces_emonet_fairface.csv", index=False)

In [ ]:
emotion_map = {
    'neutral': 'neutral',
    'happy': 'happy', 
    'sad': 'sad',
    'surprise': 'surprise',
    'fear': 'fear',
    'disgust': 'disgust',
    'anger': 'angry',
    'contempt': 'contempt'
}

data_faces['emonet_mapped'] = data_faces['emonet_emotion'].map(emotion_map)

emotion_acc = data_faces.groupby(['original_gender', 'label']).apply(
    lambda x: 100 * (x['emonet_mapped'] == x['label']).mean()
).unstack().round(1)

plt.figure(figsize=(10, 3))
sns.heatmap(emotion_acc, annot=True, fmt='.1f', cmap='RdYlGn', 
            cbar_kws={'label': 'Recall (%)'}, vmin=0, vmax=100,
            linewidths=0.5, linecolor='gray')
plt.title('EmoNet Emotion Recognition Accuracy by Gender and True Emotion')
plt.ylabel('Gender')
plt.xlabel('True Emotion')
plt.tight_layout()
plt.show()

In [ ]:
for age_group in ['young', 'middle-age', 'old']:
    data_mini = data_faces[data_faces['original_agegroup'] == age_group]

    emotion_acc = data_mini.groupby(['original_gender', 'label']).apply(
        lambda x: 100 * (x['emonet_mapped'] == x['label']).mean()
    ).unstack().round(1)
    
    plt.figure(figsize=(10, 3))
    sns.heatmap(emotion_acc, annot=True, fmt='.1f', cmap='RdYlGn', 
                cbar_kws={'label': 'Recall (%)'}, vmin=0, vmax=100,
                linewidths=0.5, linecolor='gray')
    plt.title('EmoNet Emotion Recognition Accuracy by Gender and True Emotion')
    plt.ylabel('Gender')
    plt.xlabel('True Emotion')
    plt.tight_layout()
    plt.show()

In [ ]:
data_adfes['emonet_mapped'] = data_adfes['emonet_emotion'].map(emotion_map)

emotion_acc = data_adfes.groupby(['original_gender', 'label']).apply(
    lambda x: 100 * (x['emonet_mapped'] == x['label']).mean()
).unstack().round(1)

plt.figure(figsize=(10, 3))
sns.heatmap(emotion_acc, annot=True, fmt='.1f', cmap='RdYlGn', 
            cbar_kws={'label': 'Recall (%)'}, vmin=0, vmax=100,
            linewidths=0.5, linecolor='gray')
plt.title('EmoNet Emotion Recognition Accuracy by Gender and True Emotion')
plt.ylabel('Gender')
plt.xlabel('True Emotion')
plt.tight_layout()
plt.show()